# WEEK 12｜回归、特征变换与模型选择

本周 source 延续 Scikit-learn，介绍多元线性回归、多项式特征、Pipeline、交叉验证与偏差—方差取舍，并以手写数字展示降维和分类。

## 1. 多元线性回归

简单线性回归用一个特征估计目标；多元线性回归使用多个特征。模型形式为 `y = a0 + a1*x1 + a2*x2 + ...`。fit 后可查看截距与各特征系数。

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(1)
X = 10 * rng.random((100, 3))
y = 0.5 + X @ np.array([1.5, -2.0, 1.0])

model = LinearRegression().fit(X, y)
print(model.intercept_)
print(model.coef_)

## 2. 多项式回归与 Pipeline

线性回归也可以拟合弯曲关系：先将原特征展开为 x、x²、x³ 等新特征，再对这些列拟合线性模型。PolynomialFeatures 执行特征展开；degree 决定最高幂次。

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

x = np.array([2, 3, 4])[:, None]
poly = PolynomialFeatures(degree=3, include_bias=False)
X_poly = poly.fit_transform(x)
print(X_poly)

Pipeline 把按顺序执行的步骤组成一个对象。source 用 `make_pipeline(PolynomialFeatures(...), LinearRegression())` 将特征展开与回归串起来；对 pipeline 调用 fit 时，数据依次经过各步骤。这样可以改变多项式次数并用同一流程拟合。

In [ ]:
from sklearn.pipeline import make_pipeline

def polynomial_regression(degree=2):
    return make_pipeline(
        PolynomialFeatures(degree),
        LinearRegression(),
    )

model = polynomial_regression(3).fit(x, np.array([1, 2, 3]))
print(model.predict(x))

## 3. 交叉验证与偏差—方差

validation curve 对不同超参数设置分别计算训练分数和验证分数。讲义用多项式 degree 展示模型复杂度的影响：次数太低，模型过于简单，可能无法表达数据趋势（欠拟合）；次数过高，模型可能贴合训练样本中的偶然波动（过拟合）。选择时观察训练与验证表现的变化，而不是只看训练集。

偏差描述模型过于简单造成的误差；方差描述模型对训练样本变化过于敏感造成的误差；数据本身还有不能由模型消除的噪声。讲义将模型选择说明为在偏差和方差之间取得适当平衡。

In [ ]:
rng = np.random.RandomState(1)
X_curve = rng.rand(40, 1) ** 2
y_curve = 10 - 1.0 / (X_curve.ravel() + 0.1) + rng.randn(40)
degrees = np.arange(1, 10)
train_scores, validation_scores = validation_curve(
    polynomial_regression(),
    X_curve, y_curve,
    param_name="polynomialfeatures__degree",
    param_range=degrees,
    cv=5,
)

## 4. 手写数字：降维与分类

source 加载手写数字数据并展示图像及标签。Isomap 将高维特征投影到二维，散点图按数字标签着色，用于观察数据在低维表示中的分布。

In [ ]:
from sklearn.datasets import load_digits
from sklearn.manifold import Isomap

digits = load_digits()
embedding = Isomap(n_components=2).fit_transform(digits.data)
print(digits.images.shape, digits.target.shape, embedding.shape)

分类示例将数字数据拆分训练集与测试集，用 GaussianNB 拟合训练集并预测测试标签。准确率概括预测正确的比例；混淆矩阵按真实类别和预测类别交叉统计结果，热图更直观地显示错分位置。

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix

Xtrain, Xtest, ytrain, ytest = train_test_split(
    digits.data, digits.target, random_state=0
)
classifier = GaussianNB().fit(Xtrain, ytrain)
predicted = classifier.predict(Xtest)
print(accuracy_score(ytest, predicted))
print(confusion_matrix(ytest, predicted))

### WEEK11 Scikit-learn 讲义的重复内容

本周的 `lec11_scikitlearn_Ver2` 延续并重复 WEEK11 的 Scikit-learn 工作流、Iris 分类、交叉验证、PCA 与聚类示例；相关流程见 WEEK11 精讲笔记。本讲义特别补充：比较 GaussianNB 与 RandomForestClassifier 的交叉验证得分；模型/超参数选定后，用全量数据重新训练最终模型。模型拟合后得到的 `coef_` 等是学习出的模型参数，与函数定义中的参数不同。

In [ ]:
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

iris = sns.load_dataset("iris")
X_iris = iris.iloc[:, :-1]
y_iris = iris["species"]
nb_scores = cross_val_score(GaussianNB(), X_iris, y_iris, cv=5)
rf_scores = cross_val_score(RandomForestClassifier(random_state=0), X_iris, y_iris, cv=5)
print(nb_scores.mean(), rf_scores.mean())

# 选定模型后，再使用完整数据拟合最终模型
final_model = RandomForestClassifier(random_state=0).fit(X_iris, y_iris)

## 本周回顾

多元线性回归从多个特征估计目标；多项式特征把幂次项扩展成列，使线性模型表达曲线关系；Pipeline 将转换和模型串联。交叉验证和验证曲线比较模型设置；偏差、方差与噪声帮助解释欠拟合和过拟合。手写数字示例展示 Isomap 降维和 GaussianNB 分类。